# 03 — Model Training

**Goal:** train every model to forecast **NH4, COD and TSS 4 hours ahead**, and save their
predictions so that notebook 04 can compare them fairly.

We train 8 models for each site and each indicator (3 sites x 3 indicators = 9 combinations):

| Model | Type | Idea |
|---|---|---|
| `train_mean` | baseline | always predict the average seen during training |
| **`persistence`** | **baseline** | **the value in 4 hours equals the value now** |
| `seasonal_naive` | baseline | the same clock hour yesterday |
| `moving_average` | baseline | the average of the last 4 hours |
| `drift` | baseline | continue the recent trend in a straight line |
| `ridge` | machine learning | linear model using past values |
| `gradient_boosting` | machine learning | tree model using past values |
| `lstm` | deep learning | neural network reading a 24-hour sequence |

`persistence` is the one that matters. It requires no training at all, so **any model that
cannot beat it is not worth using.** Notebook 04 tests exactly that.

**What this notebook saves** into a `predictions/` folder: one CSV per site and indicator,
holding the true value and every model's prediction for the same hours.

## Step 1 — Imports and settings

In [1]:
import json
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler

import joblib

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)
np.random.seed(42)

SITE_NAME = {"BAY_MAU": "Bay Mau", "CAU_NGA": "Cau Nga", "HO_TAY": "Ho Tay"}
SITES = list(SITE_NAME.keys())

TARGETS = ["nh4", "cod", "tss"]
HORIZON_HOURS = 4          # how far ahead we forecast
LOOKBACK_HOURS = 24        # how much history a model may look at
TEST_FRACTION = 0.15       # the most recent 15% of time is kept for testing

# Which past hours we turn into columns. 0 means "right now", 24 means "a day ago".
LAG_HOURS = [0, 1, 2, 3, 4, 6, 12, 24]

os.makedirs("predictions", exist_ok=True)
os.makedirs("models", exist_ok=True)
print("Forecasting", TARGETS, HORIZON_HOURS, "hours ahead")

Forecasting ['nh4', 'cod', 'tss'] 4 hours ahead


## Step 2 — Load the data prepared by notebook 02

In [2]:
hourly_values = {}
hourly_is_real = {}

for site in SITES:
    bundle = pd.read_pickle("processed/" + site + "_hourly.pkl")
    hourly_values[site] = bundle["values"]
    hourly_is_real[site] = bundle["is_real"]

with open("processed/selected_features.json") as file:
    selected_features = json.load(file)

for site in SITES:
    print(SITE_NAME[site], ":", len(hourly_values[site]), "hours")
print()
print("Example - features chosen for Cau Nga:")
print(selected_features["CAU_NGA"])

Bay Mau : 8784 hours
Cau Nga : 7048 hours
Ho Tay : 7048 hours

Example - features chosen for Cau Nga:
{'nh4': ['ph', 'cod', 'nh4'], 'cod': ['temp', 'tss', 'cod', 'nh4', 'no3'], 'tss': ['temp', 'ph', 'tss', 'cod']}


## Step 3 — Turn the time series into a table of past values

A model cannot read a time series directly. We build a normal table where **each row is one
moment in time**, the columns describe the past, and the answer column `y` is the value
4 hours later.

For example `nh4_lag0` is NH4 right now and `nh4_lag24` is NH4 a day ago.

In [3]:
def build_feature_table(site, target):
    """One row per hour: past values as columns, and y = the value 4 hours later."""
    values = hourly_values[site]
    features = selected_features[site][target]

    table = pd.DataFrame(index=values.index)

    # Past values of every chosen column.
    for column in features:
        for lag in LAG_HOURS:
            table[column + "_lag" + str(lag)] = values[column].shift(lag)

        # Two simple summaries of recent behaviour.
        table[column + "_mean_last_4h"] = values[column].rolling(HORIZON_HOURS).mean()
        table[column + "_change_last_4h"] = values[column] - values[column].shift(HORIZON_HOURS)

    # Time of day and day of week, in case pollution follows a daily rhythm.
    table["hour_sin"] = np.sin(2 * np.pi * values.index.hour / 24)
    table["hour_cos"] = np.cos(2 * np.pi * values.index.hour / 24)
    table["day_of_week"] = values.index.dayofweek

    # The answer we want to predict: the value HORIZON_HOURS from now.
    table["y"] = values[target].shift(-HORIZON_HOURS)

    return table


example = build_feature_table("CAU_NGA", "nh4")
print("Feature table shape:", example.shape)
example[["nh4_lag0", "nh4_lag4", "nh4_lag24", "hour_sin", "y"]].head()

Feature table shape: (7048, 34)


,nh4_lag0,nh4_lag4,nh4_lag24,hour_sin,y
datetime,,,,,
2024-03-13 08:00:00,0.690000,NaN,NaN,8.660254e-01,0.695833
2024-03-13 09:00:00,0.690000,NaN,NaN,7.071068e-01,0.699167
2024-03-13 10:00:00,0.690000,NaN,NaN,5.000000e-01,0.699167
2024-03-13 11:00:00,0.690000,NaN,NaN,2.588190e-01,0.690833
2024-03-13 12:00:00,0.695833,0.69,NaN,1.224647e-16,0.677500


## Step 4 — Decide which rows we are allowed to use

This is the step that keeps the results honest. Remember from notebook 01 that Ho Tay is missing
52 days, and those hours were filled in by us, not measured.

A row is usable only if **all five** conditions hold:

1. the hour we are predicting (4 hours ahead) was really measured
2. the current hour was really measured — `persistence` needs it
3. the hour 24 hours ago was really measured — `seasonal_naive` needs it
4. none of the feature columns are missing
5. the full 24-hour history is available — the LSTM needs it

Conditions 3 and 5 are included for *every* model, not just the ones that need them. If we did
not do this, some models would be tested on easier rows than others and the comparison would
be unfair.

In [4]:
def find_usable_rows(site, target, table):
    """Return True/False for every row, saying whether we may use it."""
    is_real = hourly_is_real[site][target]
    features = selected_features[site][target]

    # 1. the hour we are predicting was really measured
    target_is_real = is_real.shift(-HORIZON_HOURS).fillna(False).astype(bool)

    # 2. the current hour was really measured
    now_is_real = is_real.fillna(False).astype(bool)

    # 3. the same hour yesterday was really measured
    yesterday_is_real = is_real.shift(24).fillna(False).astype(bool)

    # 4. no feature column is missing
    features_complete = table.notna().all(axis=1)

    # 5. the whole 24-hour history is available
    all_present = hourly_values[site][features].notna().all(axis=1).astype(int)
    history_complete = all_present.rolling(LOOKBACK_HOURS).min().eq(1).fillna(False)

    return (target_is_real & now_is_real & yesterday_is_real
            & features_complete & history_complete)


# Show how much data survives the rule.
check = []
for site in SITES:
    for target in TARGETS:
        table = build_feature_table(site, target)
        usable = find_usable_rows(site, target, table)
        check.append({"site": SITE_NAME[site], "indicator": target,
                      "all_hours": len(table), "usable_hours": int(usable.sum())})

print("How many hours can honestly be used?")
pd.DataFrame(check).set_index(["site", "indicator"])

How many hours can honestly be used?


all_hours  usable_hours
site    indicator                         
Bay Mau nh4             8784          2755
        cod             8784          2514
        tss             8784          2915
Cau Nga nh4             7048          5324
        cod             7048          5324
        tss             7048          5324
Ho Tay  nh4             7048          2911
        cod             7048          2943
        tss             7048          2268

## Step 5 — Split into training and testing periods

We split by **time**, not randomly. The training set is the earlier part of the year and the
test set is the most recent 15%. Splitting randomly would let a model learn from the future,
which it could never do in reality.

In [5]:
def split_by_time(site, usable):
    """Earlier 85% of the calendar is for training, the last 15% for testing."""
    all_times = hourly_values[site].index
    split_time = all_times[int(len(all_times) * (1 - TEST_FRACTION))]

    is_training = pd.Series(all_times < split_time, index=all_times)

    train_rows = usable & is_training
    test_rows = usable & (~is_training)
    return train_rows, test_rows, split_time


# Quick look at one case.
table = build_feature_table("CAU_NGA", "nh4")
usable = find_usable_rows("CAU_NGA", "nh4", table)
train_rows, test_rows, split_time = split_by_time("CAU_NGA", usable)
print("Split date:", split_time)
print("Training hours:", int(train_rows.sum()))
print("Testing hours :", int(test_rows.sum()))

Split date: 2024-11-17 22:00:00
Training hours: 4458
Testing hours : 866


## Step 6 — The baseline models

None of these are trained. They are simple rules, and they are the standard every real model
must beat.

In [6]:
def baseline_predictions(site, target, train_rows):
    """The five simple rules. Each is indexed at time t and predicts time t + 4h."""
    series = hourly_values[site][target]
    answer = series.shift(-HORIZON_HOURS)

    predictions = {}

    # Always guess the average of the training period.
    predictions["train_mean"] = pd.Series(answer[train_rows].mean(), index=series.index)

    # Nothing will change in the next 4 hours.
    predictions["persistence"] = series

    # It will be whatever it was at this clock hour yesterday.
    # We want the value at (t + 4h - 24h), which is the value 20 hours before t.
    predictions["seasonal_naive"] = series.shift(24 - HORIZON_HOURS)

    # The average of the last 4 hours, which is persistence with less noise.
    predictions["moving_average"] = series.rolling(HORIZON_HOURS).mean()

    # Continue the recent trend: add the change of the last 4 hours again.
    predictions["drift"] = series + (series - series.shift(HORIZON_HOURS))

    return predictions

## Step 7 — The machine learning models

`ridge` is a linear model and `gradient_boosting` builds many small decision trees. Both read
the table of past values from Step 3.

The scaler is fitted on the **training rows only**. Fitting it on all the data would leak
information about the test period into training.

In [7]:
def machine_learning_predictions(table, train_rows, test_rows):
    """Train ridge and gradient boosting.

    Returns both the predictions AND the fitted models, so that Step 10 can
    save whichever one notebook 04 ends up choosing."""
    feature_columns = [c for c in table.columns if c != "y"]

    x_train = table.loc[train_rows, feature_columns]
    y_train = table.loc[train_rows, "y"]

    # Scale for the linear model, using training data only.
    scaler = StandardScaler().fit(x_train)

    ridge = Ridge(alpha=1.0).fit(scaler.transform(x_train), y_train)

    boosting = HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.05, max_depth=6,
        early_stopping=True, validation_fraction=0.15, random_state=42,
    ).fit(x_train, y_train)

    x_test = table.loc[test_rows, feature_columns]

    predictions = {
        "ridge": pd.Series(ridge.predict(scaler.transform(x_test)), index=x_test.index),
        "gradient_boosting": pd.Series(boosting.predict(x_test), index=x_test.index),
    }

    fitted = {
        "ridge": ridge,
        "gradient_boosting": boosting,
        "scaler": scaler,
        "feature_columns": feature_columns,   # exact column order used in training
    }
    return predictions, fitted

## Step 8 — The LSTM

The LSTM reads the last 24 hours as a sequence instead of as separate columns. It is given the
same features, the same split and the same usable rows as everything else, so the comparison in
notebook 04 is fair.

If TensorFlow is not installed the notebook simply skips this model and carries on.

In [8]:
try:
    from tensorflow import keras
    from tensorflow.keras import layers
    keras.utils.set_random_seed(42)
    HAVE_TENSORFLOW = True
    print("TensorFlow found - the LSTM will be trained.")
except ModuleNotFoundError:
    HAVE_TENSORFLOW = False
    print("TensorFlow not installed - skipping the LSTM.")
    print("Install it with:   pip install tensorflow")

TensorFlow found - the LSTM will be trained.


In [9]:
def make_sequences(site, target, rows):
    """Turn the chosen rows into 3D blocks of shape (rows, 24 hours, features)."""
    values = hourly_values[site]
    features = selected_features[site][target]
    matrix = values[features].values

    # Position number of each timestamp, so we can slice the 24 hours before it.
    position_of = {time: i for i, time in enumerate(values.index)}

    blocks = []
    for time in values.index[rows]:
        end = position_of[time]
        blocks.append(matrix[end - LOOKBACK_HOURS + 1: end + 1])
    return np.array(blocks)


def lstm_predictions(site, target, table, train_rows, test_rows):
    """Train a small LSTM and predict on the test rows."""
    x_train = make_sequences(site, target, train_rows)
    x_test = make_sequences(site, target, test_rows)
    y_train = table.loc[train_rows, "y"].values

    # Scale using the training sequences only.
    flat = x_train.reshape(-1, x_train.shape[-1])
    average = flat.mean(axis=0)
    spread = flat.std(axis=0) + 1e-8
    x_train = (x_train - average) / spread
    x_test = (x_test - average) / spread

    y_average = y_train.mean()
    y_spread = y_train.std() + 1e-8

    model = keras.Sequential([
        layers.Input(shape=x_train.shape[1:]),
        layers.LSTM(64),
        layers.Dropout(0.1),
        layers.Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])

    stop_early = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True)

    model.fit((x_train), (y_train - y_average) / y_spread,
              validation_split=0.15, epochs=60, batch_size=64,
              verbose=0, callbacks=[stop_early])

    scaled = model.predict(x_test, verbose=0).ravel()
    return pd.Series(scaled * y_spread + y_average,
                     index=table.loc[test_rows].index)

## Step 9 — Train everything and save the predictions

For each site and indicator we build one CSV containing the true answer and every model's
prediction, all for exactly the same hours.

In [10]:
fitted_models = {}          # keep the trained models so Step 10 can save them

for site in SITES:
    for target in TARGETS:
        table = build_feature_table(site, target)
        usable = find_usable_rows(site, target, table)
        train_rows, test_rows, split_time = split_by_time(site, usable)

        # Start the result with the true answer on the test hours.
        result = pd.DataFrame({"actual": table.loc[test_rows, "y"]})

        # Baselines.
        for name, series in baseline_predictions(site, target, train_rows).items():
            result[name] = series[test_rows]

        # Machine learning.
        ml_predictions, fitted = machine_learning_predictions(table, train_rows, test_rows)
        for name, series in ml_predictions.items():
            result[name] = series
        fitted_models[(site, target)] = fitted

        # Deep learning.
        if HAVE_TENSORFLOW:
            result["lstm"] = lstm_predictions(site, target, table, train_rows, test_rows)

        result.index.name = "forecast_made_at"
        result.to_csv("predictions/" + site + "_" + target + ".csv")

        print(SITE_NAME[site], target,
              "-> trained on", int(train_rows.sum()), "hours,",
              "saved", len(result), "test predictions")

Bay Mau nh4 -> trained on 2083 hours, saved 672 test predictions


Bay Mau cod -> trained on 2144 hours, saved 370 test predictions


Bay Mau tss -> trained on 2217 hours, saved 698 test predictions


Cau Nga nh4 -> trained on 4458 hours, saved 866 test predictions


Cau Nga cod -> trained on 4458 hours, saved 866 test predictions


Cau Nga tss -> trained on 4458 hours, saved 866 test predictions


Ho Tay nh4 -> trained on 2686 hours, saved 225 test predictions


Ho Tay cod -> trained on 2711 hours, saved 232 test predictions


Ho Tay tss -> trained on 2099 hours, saved 169 test predictions


In [11]:
# Check that nothing is missing in the saved files.
example = pd.read_csv("predictions/CAU_NGA_nh4.csv", index_col=0, parse_dates=True)
print("Columns saved:", list(example.columns))
print("Any missing values?", example.isna().any().any())
example.head()

Columns saved: ['actual', 'train_mean', 'persistence', 'seasonal_naive', 'moving_average', 'drift', 'ridge', 'gradient_boosting', 'lstm']
Any missing values? False


,actual,train_mean,persistence,seasonal_naive,moving_average,drift,ridge,gradient_boosting,lstm
forecast_made_at,,,,,,,,,
2024-11-17 22:00:00,1.605833,0.745376,1.567500,1.558333,1.553125,1.600000,1.566166,1.588334,1.581950
2024-11-17 23:00:00,1.604167,0.745376,1.576667,1.565000,1.562708,1.615000,1.570465,1.584985,1.589825
2024-11-18 00:00:00,1.621667,0.745376,1.584167,1.568333,1.572083,1.621667,1.583874,1.589920,1.598610
2024-11-18 01:00:00,1.635000,0.745376,1.588333,1.571667,1.579167,1.616667,1.581657,1.595258,1.606647
2024-11-18 02:00:00,1.643333,0.745376,1.605833,1.574167,1.588750,1.644167,1.591850,1.608635,1.617205


## Step 10 — Save the fitted models

The baselines (`persistence`, `moving_average` and the rest) are plain rules with nothing to
store — `forecast.py` recreates them in one line each. Only the machine-learning models have
trained parameters that must be written to disk.

Each saved file also records **the exact feature column order used during training**. When
`forecast.py` builds features later it reindexes to this list, so a mismatch fails loudly
instead of silently feeding the model columns in the wrong order.

In [12]:
for (site, target), fitted in fitted_models.items():
    for model_name in ["ridge", "gradient_boosting"]:
        bundle = {
            "model": fitted[model_name],
            "model_name": model_name,
            "site": site,
            "indicator": target,
            "feature_columns": fitted["feature_columns"],
            "features": selected_features[site][target],
            "lag_hours": LAG_HOURS,
            "horizon_hours": HORIZON_HOURS,
        }
        # The scaler is only meaningful for ridge.
        if model_name == "ridge":
            bundle["scaler"] = fitted["scaler"]

        path = "models/" + site + "_" + target + "_" + model_name + ".pkl"
        joblib.dump(bundle, path)

print("Saved", len(fitted_models) * 2, "fitted models into models/")
print()
print("Baselines are NOT saved because they have no parameters:")
print("   persistence      -> the current value")
print("   moving_average   -> the mean of the last 4 hours")
print("   seasonal_naive   -> the value 20 hours ago")
print("   drift            -> current value + recent change")

Saved 18 fitted models into models/

Baselines are NOT saved because they have no parameters:
   persistence      -> the current value
   moving_average   -> the mean of the last 4 hours
   seasonal_naive   -> the value 20 hours ago
   drift            -> current value + recent change


## Summary

| Step | What we did |
|---|---|
| 3 | turned the time series into a table of past values |
| 4 | kept only rows where the data is genuinely measured |
| 5 | split by time so no model can see the future |
| 6 | built five simple baselines, `persistence` being the key one |
| 7 | trained ridge and gradient boosting |
| 8 | trained an LSTM on the same rows |
| 9 | saved one CSV of predictions per site and indicator |
| 10 | saved the fitted machine-learning models to `models/` |

No accuracy has been measured yet. That happens next, in
**`04_evaluation_comparison.ipynb`**.